In [1]:
from bs4 import BeautifulSoup
import csv
url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"#Wikipedia will block you so just download HTML manually with browser

In [2]:
with open("https___en.wikipedia.org_wiki_2026_FIFA_World_Cup_squads.html", "r") as f:
    html=f.read()

soup = BeautifulSoup(html, 'html.parser')

'/wiki/Luis_Mej%C3%ADa'

In [14]:
# Map Wikipedia's position abbreviations to your requested full names
position_map = {
    'GK': 'Goalkeeper',
    'DF': 'Defender',
    'MF': 'Midfielder',
    'FW': 'Forward'
}

csv_data = []

current_group = None
current_team = None

# Iterate through headings and tables in the exact order they appear on the page
for element in soup.find_all(['h2', 'h3', 'table']):
    
    # 1. Update the Group if we hit an <h2>
    if element.name == 'h2':
        text = element.get_text(strip=True)
        if text.startswith('Group'):
            current_group = text.replace('Group ', '')
            
    # 2. Update the Team if we hit an <h3>
    elif element.name == 'h3':
        current_team = element.get_text(strip=True)
        
    # 3. Parse the player rows if we hit a wikitable
    elif element.name == 'table' and 'wikitable' in element.get('class', []):
        
        # Only parse if we already know the group and team context
        if not current_group or not current_team:
            continue
            
        # Wikipedia tags player rows with the 'nat-fs-player' class
        for row in element.find_all('tr', class_='nat-fs-player'):
            cols = row.find_all(['td', 'th'])
            
            # Ensure the row has the expected number of columns to prevent index errors
            if len(cols) < 7:
                continue
            
            # Extract Position (Looking specifically at the <a> tag ignores the hidden sorting spans)
            pos_tag = cols[1].find('a')
            pos_abbr = pos_tag.get_text(strip=True) if pos_tag else cols[1].get_text(strip=True)
            position = position_map.get(pos_abbr, pos_abbr)
            
            # Extract Name (Also inside an <a> tag inside a <th> element)
            name_tag = cols[2].find('a')
            name = name_tag.get_text(strip=True) if name_tag else cols[2].get_text(strip=True)
            player_url = name_tag.attrs['href']
            
            # Extract Club Team (Grabbing the last <a> tag skips the country flag icon link)
            club_links = cols[6].find_all('a')
            club_team = club_links[-1].get_text(strip=True)
            club_url=club_links[-1].attrs['href']
            
            csv_data.append([position, name, player_url, club_team, club_url, current_group, current_team])

# Write the compiled data to a CSV file
with open('2026_world_cup_team_affiliations.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['position', 'name', 'player_url', 'club_team', 'club_url', 'group', 'international_team'])
    writer.writerows(csv_data)

print(f"Successfully parsed {len(csv_data)} players to world_cup_squads.csv!")

Successfully parsed 1248 players to world_cup_squads.csv!


In [13]:
element

<h3 id="Czech_Republic">Czech Republic</h3>

In [32]:
import pandas as pd
df = pd.read_csv("2026_world_cup_team_affiliations.csv")
df

,position,name,club_team,group,international_team
0,Goalkeeper,Matěj Kovář,PSV Eindhoven,A,Czech Republic
1,Defender,David Zima,Slavia Prague,A,Czech Republic
2,Defender,Tomáš Holeš,Slavia Prague,A,Czech Republic
3,Defender,Robin Hranáč,TSG Hoffenheim,A,Czech Republic
4,Defender,Vladimír Coufal,TSG Hoffenheim,A,Czech Republic
...,...,...,...,...,...
1243,Goalkeeper,Orlando Mosquera,Al-Fayha,L,Panama
1244,Defender,Michael Amir Murillo,Beşiktaş,L,Panama
1245,Forward,Azarias Londoño,Universidad Católica,L,Panama
1246,Defender,Roderick Miller,Turan Tovuz,L,Panama


In [ ]:
k
